# 01 — Product Image Classifier Training

Transfer learning with **MobileNetV2** to classify product images into 5 categories: shoes, bags, electronics, clothing, groceries.

**Before running:** point `DATA_DIR` at a folder with one subfolder per class, e.g.:
```
data/products/
  shoes/
  bags/
  electronics/
  clothing/
  groceries/
```
Any small labeled image set works (Kaggle 'Retail Product Checkout Dataset', a scraped set, or Fashion-MNIST reorganized into folders).

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print('TF version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

## 1. Config

In [ ]:
DATA_DIR = '../data/products'   # <-- point this at your class-folder dataset
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 10
CLASS_NAMES = ['shoes', 'bags', 'electronics', 'clothing', 'groceries']

## 2. Load data with `image_dataset_from_directory`
This auto-infers labels from subfolder names and handles the train/val split.

In [ ]:
train_ds = keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset='training', seed=42,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
)
val_ds = keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset='validation', seed=42,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
)

class_names = train_ds.class_names
print('Detected classes:', class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

## 3. Data augmentation + preprocessing

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

preprocess_input = keras.applications.mobilenet_v2.preprocess_input

## 4. Build the model (MobileNetV2 base, frozen, + custom head)

In [ ]:
base_model = keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet'
)
base_model.trainable = False  # freeze for the first pass

inputs = keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(len(class_names), activation='softmax')(x)
model = keras.Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()

## 5. Train (head only, base frozen)

In [ ]:
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)

## 6. (Optional) Fine-tune: unfreeze top layers of the base model
Do this only after the head has converged — unfreezing too early destroys the pretrained features.

In [ ]:
base_model.trainable = True
fine_tune_at = len(base_model.layers) - 20
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

fine_tune_epochs = 5
history_fine = model.fit(
    train_ds, validation_data=val_ds, epochs=fine_tune_epochs
)

## 7. Evaluate

In [ ]:
loss, acc = model.evaluate(val_ds)
print(f'Validation accuracy: {acc:.4f}')

## 8. Save the model
Saved to `app/models/product_classifier.h5` — `cv_service.py` loads it from exactly this path.

In [ ]:
import os
os.makedirs('../app/models', exist_ok=True)
model.save('../app/models/product_classifier.h5')
print('Saved to ../app/models/product_classifier.h5')